In [1]:
import os
import time
import json
import shutil
import argparse
import numpy as np

import functools
import operator

import pickle, gzip

import uproot

In [2]:
def combine_vineReduce_channels(pkl): 
    vineReduce_dict = load_pickle(pkl_file)
    first_ch = next(iter(vineReduce_dict))
    variables = vineReduce_dict[first_ch].keys()
    
    hists = {}
    for var in variables: 
        h_list = [vineReduce_dict[ch][var] for ch in vineReduce_dict]
        hists[var] = functools.reduce(operator.add, h_list)

    return hists

def flatten_vineReduce_hists(pkl, outname = 'flat_hists'):
    # Flattens the histogram structure from vineReduce and saves as a new pickle. 
    # Returns the name of the new pickle file 
    
    flattend_hists = combine_vineReduce_channels(pkl)
    tmp_pkl = f"{outname}.pkl.gz"
    with gzip.open(tmp_pkl, "wb") as f:
        pickle.dump(flattend_hists, f)
    
    return tmp_pkl

def load_pickle(fname):
    return pickle.load(gzip.open(fname))

In [3]:
file_path = "test_200424_full/TESTING-ttbar-ee_2b_2j_mllbb.root" 
with uproot.open(file_path) as f:
    print(f"Inspecting file: {file_path}")

    # Get all histogram names (keys)
    all_keys = f.keys(cycle=False)

    # for key in all_keys:
    for key in ['tt_sm_elecIDUp', 'tt_sm_elecIDDown']:
        print(f"{key}")
        print(f[key].values())

Inspecting file: test_200424_full/TESTING-ttbar-ee_2b_2j_mllbb.root
tt_sm_elecIDUp
[4.55211006e+01 2.84796939e+02 1.02475672e+03 1.61534667e+03
 1.99568009e+03 2.00786482e+03 1.74225487e+03 1.45563657e+03
 1.05047310e+03 6.93460328e+02 4.09928640e+02 2.24377960e+02
 1.46109062e+02 5.31342180e+01 9.64899733e+00 6.09458530e-01]
tt_sm_elecIDDown
[4.28226925e+01 2.67450941e+02 9.60671992e+02 1.51113986e+03
 1.86362084e+03 1.87216787e+03 1.62214263e+03 1.35346229e+03
 9.75395832e+02 6.42643295e+02 3.79251822e+02 2.07108375e+02
 1.34360865e+02 4.85075189e+01 8.77034589e+00 5.48525916e-01]


In [4]:
pkl_file = "../analysis/SR_ALL_260424.pkl.gz"
prepared_pkl = flatten_vineReduce_hists(pkl_file)

In [5]:
hist_dict = load_pickle(prepared_pkl)

In [6]:
channels = list(hist_dict['mllbb'].axes['channel'])
all_procs = list(hist_dict['mllbb'].axes['process'])
systs = list(hist_dict['mllbb'].axes['systematic'])
systs.remove('sumw2')
data_procs = [x for x in all_procs if 'data' in x]
MC_procs = [x for x in all_procs if x not in data_procs]

In [21]:
tt_procs = [x for x in MC_procs if 'TT01j2l' in x]
tW_procs = [x for x in MC_procs if 'TW' in x]
DY_procs = [x for x in MC_procs if 'DY' in x]
Others_procs = [x for x in MC_procs if ((x not in tt_procs) and (x not in tW_procs) and (x not in DY_procs))]

print(DY_procs)

['DY10to50_centralUL16APV', 'DY50_centralUL16APV', 'DY10to50_centralUL16', 'DY50_centralUL16', 'DY10to50_centralUL17', 'DY50_centralUL17', 'DY10to50_centralUL18', 'DY50_centralUL18']


In [8]:
# base_procs = ['tt', 'tW', 'DY', 'Others']

grouped_procs = {
    'tt':tt_procs,
    'tW':tW_procs,
    'DY':DY_procs,
    'Others': Others_procs
}

In [20]:
# Checks that the bin values in the histogram for combine matches what I have from the orignal histogram file
# Doesn't work for systematics that are specific to the year, those are handled separately below. 

for ch in channels: 
    print(f"channel: {ch}")
    root_file = os.path.join("test_200424_full", f"TESTING-ttbar-{ch}_mllbb.root")
    with uproot.open(root_file) as f:
        print(f"Inspecting file: {root_file}")
        all_keys = f.keys(cycle=False)
        # print(all_keys)
        for p in grouped_procs.keys(): 
            print(f"  proc: {p}")
            correct_systs = []
            wrong_systs = []
            for s in systs:
                if 'fact' in s or 'renorm' in s: 
                    continue
                if ('2016' in s) or ('2017' in s) or ('2018' in s): 
                    continue
                if s == 'nominal': 
                    key_name = f"{p}_sm"
                else: 
                    key_name = f"{p}_sm_{s}"
#                 print(key_name in all_keys)
#                 if not key_name in all_keys:
#                     print(f"key_name doesn't exist: {key_name}")
                root_values = f[key_name].values()
                # print(f"\tkey: {key_name}")
                # print(f"\t root_vals: {root_values}")
                proc_list = grouped_procs[p]
                orig_hist = hist_dict['mllbb'][{'channel':ch, 'systematic':s, 'process':proc_list}][{'process':sum}].as_hist({})
                orig_values = orig_hist.values()
                # print(f"\t orig_vals: {orig_values}")
                matches = np.allclose(root_values[:15], orig_values)
                # print(f"\t    Equal? : {matches} \n")
                if matches: 
                    correct_systs.append(s)
                if not matches: 
                    wrong_systs.append(s)
            print(f"\tsysts that match: {correct_systs}")
            print(f"\tsysts that DO NOT match: {wrong_systs} \n\n")

channel: ee_2b_2j
Inspecting file: test_200424_full/TESTING-ttbar-ee_2b_2j_mllbb.root
  proc: tt
	systs that match: ['nominal', 'elecIDUp', 'elecIDDown', 'muonIDUp', 'muonIDDown', 'muonISOUp', 'muonISODown', 'trigSFUp', 'trigSFDown', 'L1prefireUp', 'L1prefireDown', 'PUUp', 'PUDown', 'jetPuIDUp', 'jetPuIDDown', 'btagSFbc_correlatedUp', 'btagSFbc_correlatedDown', 'btagSFlight_correlatedUp', 'btagSFlight_correlatedDown', 'FSRUp', 'FSRDown', 'ISRUp', 'ISRDown', 'hdampUp', 'hdampDown', 'JES_AbsoluteStatUp', 'JES_AbsoluteStatDown', 'JES_AbsoluteScaleUp', 'JES_AbsoluteScaleDown', 'JES_AbsoluteMPFBiasUp', 'JES_AbsoluteMPFBiasDown', 'JES_FragmentationUp', 'JES_FragmentationDown', 'JES_SinglePionECALUp', 'JES_SinglePionECALDown', 'JES_SinglePionHCALUp', 'JES_SinglePionHCALDown', 'JES_FlavorQCDUp', 'JES_FlavorQCDDown', 'JES_FlavorPureBottomUp', 'JES_FlavorPureBottomDown', 'JES_FlavorPureCharmUp', 'JES_FlavorPureCharmDown', 'JES_FlavorPureGluonUp', 'JES_FlavorPureGluonDown', 'JES_TimePtEtaUp', 'JE